In [3]:
import numpy as np
import pandas as pd
from keras.models import *
from keras.layers import *
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle

In [4]:
# dataset = pd.read_csv('sentiment_ds.csv')
dataset = pd.read_csv('sentiment_ds.csv', encoding='latin1')

In [5]:
dataset

,phrase,sentiment
0,"""I love spending time with my family.""",positive
1,"""Sunshine always brightens my day.""",positive
2,"""Helping others is so rewarding.""",positive
3,"""A good book can transport you to another world.""",positive
4,"""The smell of freshly baked bread is amazing.""",positive
...,...,...
3474,"""I am krituk""",neutral
3475,"""I am Jayesh Chak""",neutral
3476,"""Your face is ugly""",negative
3477,"""Today is a very Lovely outside.""",positive


In [6]:
dataset.shape

(3479, 2)

In [7]:
dataset.isnull().sum()

phrase       0
sentiment    0
dtype: int64

In [8]:
dataset['sentiment'].unique()

array(['positive', 'negative', 'neutral', "positive'"], dtype=object)

In [9]:
# Change all occurrences of 'Alice' to 'Alex' in the 'Name' column
dataset['sentiment'] = dataset['sentiment'].replace("positive'", 'positive')
dataset


,phrase,sentiment
0,"""I love spending time with my family.""",positive
1,"""Sunshine always brightens my day.""",positive
2,"""Helping others is so rewarding.""",positive
3,"""A good book can transport you to another world.""",positive
4,"""The smell of freshly baked bread is amazing.""",positive
...,...,...
3474,"""I am krituk""",neutral
3475,"""I am Jayesh Chak""",neutral
3476,"""Your face is ugly""",negative
3477,"""Today is a very Lovely outside.""",positive


In [10]:
dataset['sentiment'].unique()

array(['positive', 'negative', 'neutral'], dtype=object)

In [11]:
dataset.isnull().sum()

phrase       0
sentiment    0
dtype: int64

In [12]:
dataset['sentiment'].value_counts()

sentiment
positive    1291
negative    1129
neutral     1059
Name: count, dtype: int64

In [13]:
label_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

dataset['label'] = dataset['sentiment'].map(label_map)

In [14]:
dataset

,phrase,sentiment,label
0,"""I love spending time with my family.""",positive,2
1,"""Sunshine always brightens my day.""",positive,2
2,"""Helping others is so rewarding.""",positive,2
3,"""A good book can transport you to another world.""",positive,2
4,"""The smell of freshly baked bread is amazing.""",positive,2
...,...,...,...
3474,"""I am krituk""",neutral,1
3475,"""I am Jayesh Chak""",neutral,1
3476,"""Your face is ugly""",negative,0
3477,"""Today is a very Lovely outside.""",positive,2


In [15]:
duplicate = dataset[dataset.duplicated()]

In [16]:
duplicate

,phrase,sentiment,label


In [17]:
dataset.drop_duplicates(keep=False,inplace=True)

In [18]:
dataset

,phrase,sentiment,label
0,"""I love spending time with my family.""",positive,2
1,"""Sunshine always brightens my day.""",positive,2
2,"""Helping others is so rewarding.""",positive,2
3,"""A good book can transport you to another world.""",positive,2
4,"""The smell of freshly baked bread is amazing.""",positive,2
...,...,...,...
3474,"""I am krituk""",neutral,1
3475,"""I am Jayesh Chak""",neutral,1
3476,"""Your face is ugly""",negative,0
3477,"""Today is a very Lovely outside.""",positive,2


In [19]:
dataset=shuffle(dataset)

In [20]:
dataset

,phrase,sentiment,label
962,"""I had a disagreement with a friend, but it de...",positive,2
2720,"""The car is red.""",neutral,1
1835,"""It is possible that he may drop and break his...",negative,0
394,"""I'm blessed with wonderful friends.""",positive,2
759,"""I will be delighted by that heartwarming surp...",positive,2
...,...,...,...
998,"""I can't believe it's still raining.""",negative,0
2330,"""Advanced mathematics plays a pivotal role in ...",neutral,1
3318,"""Sharing a meal with loved ones, whether it's ...",positive,2
2154,"""People talk to communicate.""",neutral,1


In [21]:
X_data=dataset['phrase'].values
labels=dataset['label'].values

In [22]:
X_data.shape,labels.shape

((3479,), (3479,))

In [23]:
split=0.2
X_train=X_data[(int)(split*X_data.shape[0]):]
X_test=X_data[:(int)(split*X_data.shape[0])]
Y_train=labels[int(split*labels.shape[0]):]
Y_test=labels[:int(split*labels.shape[0])]

In [24]:
print(X_train.shape,X_test.shape,Y_train.shape,Y_test.shape)

(2784,) (695,) (2784,) (695,)


In [25]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3479 entries, 962 to 2663
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   phrase     3479 non-null   object
 1   sentiment  3479 non-null   object
 2   label      3479 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 108.7+ KB


In [26]:
np.unique(labels)

array([0, 1, 2])

In [27]:
Y_test=to_categorical(Y_test)
Y_train=to_categorical(Y_train)

In [28]:
print(X_train.shape,X_test.shape,Y_train.shape,Y_test.shape)

(2784,) (695,) (2784, 3) (695, 3)


In [29]:
# f=open('D:/data science CT/Projects/data_science___project/sentiment_analysis/glove.6B.50d.txt',encoding='utf-8')
f = open("glove.6B/glove.6B.50d.txt", encoding="utf-8")

embedding_index={}

for line in f:
    values=line.split()
    word=values[0]
    coefs=np.asarray(values[1:],dtype='float')
    embedding_index[word]=coefs
    
f.close()

In [30]:
def embedding_output(X):
    maxLen=50 #max words in sentence
    embedding_out=np.zeros((X.shape[0],maxLen,50))
    for ix in range(X.shape[0]):
        # print(ix)
        try:
            X[ix]=X[ix].split()
        except:
            pass
        for ij in range(maxLen):
            try:
                embedding_out[ix][ij]=embedding_index[X[ix][ij].lower()]
            except:
                embedding_out[ix][ij]=np.zeros((50,))
    return embedding_out

In [31]:
X_train=embedding_output(X_train)
X_test=embedding_output(X_test)

In [32]:
X_train.shape,Y_train.shape,X_test.shape,Y_test.shape

((2784, 50, 50), (2784, 3), (695, 50, 50), (695, 3))

In [33]:
model=Sequential()
model.add(LSTM(50,input_shape=(50,50),return_sequences=True))
model.add(Dropout(0.1))
model.add(LSTM(20,return_sequences=True))
model.add(Dropout(0.1))
model.add(LSTM(10,return_sequences=False))
model.add(Dropout(0.1))
model.add(Dense(3,activation='softmax'))
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['acc'])

C:\Users\TANISHK\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [34]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 50, 50)              │          20,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 50, 50)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 50, 20)              │           5,680 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 50, 20)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 10)                  │           1,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 10)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 3)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 27,153 (106.07 KB)

 Trainable params: 27,153 (106.07 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
hist=model.fit(X_train,Y_train,epochs=25,batch_size=50,shuffle=True,validation_split=0.2)

Epoch 1/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 8s 55ms/step - acc: 0.4266 - loss: 1.0637 - val_acc: 0.4740 - val_loss: 1.0077
Epoch 2/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - acc: 0.5635 - loss: 0.9126 - val_acc: 0.5637 - val_loss: 0.8909
Epoch 3/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - acc: 0.6062 - loss: 0.8483 - val_acc: 0.5709 - val_loss: 0.8920
Epoch 4/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - acc: 0.6291 - loss: 0.7987 - val_acc: 0.6032 - val_loss: 0.8159
Epoch 5/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - acc: 0.6583 - loss: 0.7577 - val_acc: 0.6320 - val_loss: 0.8005
Epoch 6/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - acc: 0.6969 - loss: 0.7118 - val_acc: 0.6804 - val_loss: 0.7421
Epoch 7/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - acc: 0.7544 - loss: 0.6336 - val_acc: 0.7253 - val_loss: 0.6852
Epoch 8/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - acc: 0.7795 - loss: 0.5881 - val_acc: 0.7253 - val_loss: 0.6819
Epoch 9/25
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - acc: 0.8145

In [36]:
model.evaluate(X_test,Y_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - acc: 0.8835 - loss: 0.3786


[0.3785609304904938, 0.8834532499313354]

In [37]:
model.save('model.h5')

In [38]:
# def predict(X):
#     X=np.asarray([X])
#     print(X.shape)
#     X=embedding_output(X)
#     print(model.predict(X))
#     return np.argmax(model.predict(X))

def predict(text):
    # Split the input string into a list of words
    words = text.split()
    
    # Create an empty embedding matrix for 1 sample, maxLen=50, embedding_dim=50
    embedding_out = np.zeros((1, 50, 50)) 
    
    # Map words to their GloVe embeddings
    for ij in range(min(50, len(words))):
        try:
            embedding_out[0][ij] = embedding_index[words[ij].lower()]
        except:
            embedding_out[0][ij] = np.zeros((50,))
            
    # Get the model prediction
    pred_probs = model.predict(embedding_out)
    print("Probabilities:", pred_probs)
    
    # Return the index of the highest probability
    return np.argmax(pred_probs)

In [39]:
sent='He is a good student'
sent1='He is the bad student'
sent2='he is ok student'

In [40]:
dataset

,phrase,sentiment,label
962,"[""I, had, a, disagreement, with, a, friend,, b...",positive,2
2720,"[""The, car, is, red.""]",neutral,1
1835,"[""It, is, possible, that, he, may, drop, and, ...",negative,0
394,"[""I'm, blessed, with, wonderful, friends.""]",positive,2
759,"[""I, will, be, delighted, by, that, heartwarmi...",positive,2
...,...,...,...
998,"[""I, can't, believe, it's, still, raining.""]",negative,0
2330,"[""Advanced, mathematics, plays, a, pivotal, ro...",neutral,1
3318,"[""Sharing, a, meal, with, loved, ones,, whethe...",positive,2
2154,"[""People, talk, to, communicate.""]",neutral,1


In [41]:
print(predict(sent))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step
Probabilities: [[0.00558365 0.06742732 0.9269891 ]]
2


In [42]:
print(predict(sent1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Probabilities: [[0.97874296 0.01400805 0.00724902]]
0


In [43]:
print(predict(sent2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Probabilities: [[0.31030834 0.27433828 0.4153534 ]]
2


In [44]:
model=load_model('model.h5')

In [45]:
import joblib
joblib.dump(model,'sentiment_model.pkl')